# Getting Started with DNN

Before we move to a more sophisticated ex. we will get familiar with
Tensorflow and Keras. We will take a look at a first example of
a neural network, which makes use of the Python library Keras to learn
to classify hand-written digits. The problem we are trying to solve here
is to classify grayscale images of handwritten digits (28 pixels by 28
pixels), into their 10 categories (0 to 9). The dataset we will use is
the MNIST dataset. It's a set of 60,000 training images, plus 10,000
test images, assembled by the National Institute of Standards and
Technology (the NIST in MNIST) in the 1980s. You can think of "solving"
MNIST as the "Hello World" of deep learning -- it's what you do to
verify that your algorithms are working as expected.

Firstly, we will import Keras. Historically, there were many ways to do it:

In [3]:
import keras
import tensorflow.keras as keras
from tensorflow import keras
# from tensorflow.keras import XXX

#All of the above should work. But if you start using different ways in one script it will often crash

If you are using the newest version of Keras 3 you can use it with different backends instead of just Tensorflow. Keras now supports both PyTorch and Jax as well. You should specify the backend you want, before importing Keras. Or just use a default tensorflow.

In [ ]:
# import os
# os.environ["KERAS_BACKEND"] = "tensorflow"
# import keras
# print(keras.__version__)

The MNIST dataset comes pre-loaded in Keras, in the form of a set of
four Numpy arrays:

In [4]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


**DNN workflow**
- Build the neural network architecture.
- Train our neural network with the training data, `train_images` and
  `train_labels`. The network will then learn to associate images and
  labels.
- The network will produce predictions for `test_images`, and we will
  verify if these predictions match the labels from `test_labels`.

**DNN architecture**

- Our network will consist of a sequence of two Dense layers, which are
densely-connected (also called "fully-connected") neural layers.

- The second (and last) layer is a 10-way "softmax" layer, which means
  it will return an array of 10 probability scores (summing to 1). Each
  score will be the probability that the current digit image belongs to
  one of our 10 digit classes.

**DNN training**

To make our network ready for training, we need to pick three more
things, as part of the "compilation" step:

- A **loss function**: This is how the network will be able to measure
how good a job it is doing on its training data, and thus how it will be
able to steer itself in the right direction.
- An **optimizer**: this is the mechanism through which the network will
  update itself based on the data it sees and its loss function.
- **Metrics**: to monitor during training and testing. Here we will only
  care about accuracy (the fraction of the images that were correctly
  classified).

Today, during our 3 examples we will use only the `Sequential class`.
During our next meetings, I will introduce the functional API where we
will be able to manipulate the data tensors that the model processes and
apply layers to this tensors as if they were functions.




# Network architecture

In [5]:
from keras import models
from keras import layers

network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,)))
network.add(layers.Dense(10, activation='softmax'))

#Instead of input_shape parameter, you can use an Input layer

/home/wiktor/university/sem7/practical_aspects_of_deep_learning/lab1/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1791300884.926942   18717 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4620 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5


## Network Training

Once the model architecture is defined, the learning process can be
configured in the compilation step. We specify the optimizer, loss
function, and metrics.

In [6]:

network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

**Data preparation**\
Before training, we will preprocess our data by reshaping it into the
shape that the network expects, and scaling it so that all values are in
the \[0, 1\] interval. Our training images are being stored in an array
of shape (60000, 28, 28) of type uint8 with values in the \[0, 255\]
interval. Please transform it into a float32 array of shape (60000, 28
\* 28) with values between 0 and 1.

Necessary functions: `reshape` and `astype`.

Perform it for both train and test examples.

In [9]:
import numpy as np

def prepare_data(images: np.ndarray) -> np.ndarray:
    return (
        images.reshape(images.shape[0], images.shape[1] * images.shape[2])
                .astype('float32') / 255
    )

In [10]:
train_images = prepare_data(train_images)
test_images = prepare_data(test_images)

Because of we use `categorical_crossentropy` loss function we need to
convert data format:

In [11]:
from keras.utils import to_categorical

train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

**Fit the model**\
To train the network we call the `fit` method of the network with
parameters `epochs` and `batch_size`. Set `epochs` to 5, and
`batch_size` to 128.

In [12]:
network.fit(train_images, train_labels, batch_size=128, epochs=5)

Epoch 1/5


W0000 00:00:1791301886.141191   18717 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.
W0000 00:00:1791301886.272951   18717 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.
I0000 00:00:1791301886.963252   19557 service.cc:153] XLA service 0x73439c02e770 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791301886.963271   19557 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1791301887.010523   19557 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791301887.304519   19557 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1791301887.328023   19557 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_776__.16


 23/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7378 - loss: 0.8988

I0000 00:00:1791301889.076439   19557 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


450/469 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9220 - loss: 0.2681

I0000 00:00:1791301890.538986   19557 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_776__.16


469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9233 - loss: 0.2632
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9685 - loss: 0.1059
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9786 - loss: 0.0703
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9847 - loss: 0.0501
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9889 - loss: 0.0373


## Network evaluation

Two quantities are being displayed during training: the "loss" of the
network over the training data, and the accuracy of the network over the
training data. We quickly reach an accuracy of 0.989 (i.e. 98.9%) on the
training data. Now let's check that our model performs well on the test
set too:

In [13]:
test_loss, test_acc = network.evaluate(test_images, test_labels)

print('test_acc:', test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9797 - loss: 0.0649
test_acc: 0.9797000288963318


Our test set accuracy turns out to be 97.8% -- that's quite a bit lower
than the training set accuracy.\
This gap between training accuracy and test accuracy is an example of
"overfitting", the fact that machine learning models tend to perform
worse on new data than on their training data.